
# Amari-Chentsov Updates 

Elastic Weight Consolidation (EWC) and Low Rank Adapters (LoRA) technology [1] show great potential for tuning AIs toward specialization, 
which would be useful in personalization, remote autonomy, and reducing overall compute burdens while updating models. 
These methods ultimately work by estimating the model's Fisher Information Matrix (FIM), 
which is often done in a continual or moving average fashion. 
A moving average has a solid chance to remain sufficiently up-to-date. 
However, this work explores whether or not additional first-order Taylor series adjustments can help keep the FIM estimate more-accurate. 
Provided updates occur in batches of data, or auxiliary scores can be Gaussianized over time, we may invoke the Central Limit Theorem (CLT) to enjoy approximately Gaussian-distributed steps over the statistical manifold. 
In this case, the first-order Taylor Series adjustment is done by estimating the Amari-Chentsov tensor [2] or its derivatives. 

Model choice is heavily influenced by efficiency concepts:
1. By packing all data into a sufficient statistic (instead of discarding it) and transporting it, we maximize information applied to each point visited on the statistical manifold, giving us best-possible statistical efficiency. 
2. Gaussianizing average scores gives us access to transports estimatable via scores alone, no per-observation Hessians must be calculated, saving on compute costs. 
3. We formalize a single sample-sized batch theory, thus only requiring we handle the current observation and sufficient statistics in memory. This gets us the computational efficiency of an online learning paradigm. 

TODO many terms used before their definitions - plz fix! 

## Math sketch 

EWC is applied to a continual learning context, where data are obtained from a series of independent but not necessarily identically distributed observations $X_i \sim f_X(x; \theta_i)$. 
Again, assuming each $\theta_i$ enjoys an $iid$ batch of observations, we may invoke CLT and get the following Taylor Series-based approximation.

$$ \left[ \mathcal I (\theta + d\theta) \right]_{ij} = \mathbb E_{\theta + d\theta} \partial_i \ell \partial_j \ell \approx \mathbb E_\theta \partial_i \ell \partial_j \ell + \mathbb E_\theta \partial_i \ell \partial_j \ell \partial_k \ell d\theta^k = \left[ \mathcal I (\theta) + C : d\theta \right]_{ij} $$

This is exact for canonical exponential families. More generally, our single-observation construction below applies the same first-order parameter-shift approximation to an auxiliary averaged-score process whose Gaussian limit is approximately canonical. 

where 
- $\partial_i := \partial / (\partial \theta_i)$, 
- $\ell := \log f_X(X; \theta)$, 
- $C_{ijk} := \mathbb E_\theta \partial_i \ell \partial_j \ell \partial_k \ell$ (the Amari-Chentsov tensor), 
- $a_i b^i := \sum_i a_i b_i$ (Einstein summation notation), and 
- $C : v := C_{ijk} v^k $. 

## Conclusion: Halting progress

Utilizing the Amari-Chentsov tensor requires either handling per-observation Hessians or estimating the inverse FIM. 
1. If we study a Gaussianized auxiliary score process, then conveniently $\mathbb E_\theta [ (\partial_k s_i) s_j + s_i (\partial_k s_j)] = 0$, but this requires canonical exponential family parameterization, achieved by parameterizing with $\mathcal N ( 0 , \mathcal I^{-1}(\theta))$. That inverse introduces both estimation and numerical challenges. 
2. If we study a non-Gaussianized auxiliary score process, then we must estimate $\mathbb E_\theta [ (\partial_k s_i) s_j ]$ which includes a Hessian per observation. This computational cost is _massive_.

## Why the Amari-Chentsov tensor is enough 

In the original sample model,

$$ \partial_k [\mathcal I(\theta)]_{ij} = \mathbb E_\theta [ (\partial_k s_i) s_j + s_i (\partial_k s_j) + s_i s_j s_k ]. $$

So, the Amari-Chentsov tensor alone is not enough unless the first two terms vanish, as they do in canonical exponential families. Our workaround is to treat sufficient statistic estimation as a parallel auxiliary process rather than as the raw deep learning task itself. For large batches this auxiliary process is the normalized batch score, and for single-observation updates it is the normalized exponential moving average (EMA) score below. 

By CLT, or weighted CLT for EMAs, together with local freezing via Lipschitz continuity, the auxiliary score process is approximately mean-zero Gaussian with covariance $\mathcal I(\theta)$. Parameterizing this centered Gaussian family by precision produces a canonical exponential family on the auxiliary process. In that auxiliary family the first-order parameter-shift is score-only, so the analogue of $\mathbb E_\theta [ (\partial_k s_i) s_j + s_i (\partial_k s_j) ]$ is absorbed into a remainder term $B_n$ or $B_\delta$ satisfying $\| B_n(\theta, u) \| \to 0$ or $\| B_\delta(\theta, u) \| \to 0$ in the regimes developed below. 

Thus, the Amari-Chentsov tensor is enough for the auxiliary estimation process, and approximately enough for the underlying task model through this vanishing remainder. 

This is valuable because $\mathbb E_\theta (\partial_k s_i) s_j$ includes a per-sample Hessian, 
massively increasing computational burdens. 
Instead, we utilize a framework that allows us to work only with scores $s_i$. 

It is here that the relationship between RL and Information Geometry is formalized. 
With an always-defined Fisher metric and choosen connection, a complete geometry is specified. 

## Shift operator similarity to e-connection 

What is being moved here is not the parameter itself, but the Fisher metric tensor field $\mathcal I$ over the statistical manifold. Given a small displacement $d\theta \in T_\theta \Theta$, the geometric question is how to compare $\mathcal I(\theta + d\theta)$ with $\mathcal I(\theta)$. Under the e-connection, the natural comparison is to e-parallel transport $\mathcal I(\theta + d\theta)$ back to $T_\theta \Theta$ and then expand. Writing $P_{\theta + d\theta \to \theta}^{(e)}$ for this transport, we get the first-order identity

$$ P_{\theta + d\theta \to \theta}^{(e)} \mathcal I(\theta + d\theta) = \mathcal I(\theta) + \nabla_{d\theta}^{(e)} \mathcal I(\theta) + O(\|d\theta\|^2). $$

In e-affine coordinates the e-connection coefficients vanish, so covariant differentiation reduces to ordinary differentiation. Hence

$$ [\nabla_{d\theta}^{(e)} \mathcal I(\theta)]_{ij} = d\theta^k \partial_k [\mathcal I(\theta)]_{ij}. $$

For a canonical exponential family, $d\theta^k \partial_k [\mathcal I(\theta)]_{ij} = C_{ijk} d\theta^k$, so

$$ P_{\theta + d\theta \to \theta}^{(e)} \mathcal I(\theta + d\theta) = \mathcal I(\theta) + C : d\theta + O(\|d\theta\|^2). $$

Thus, to first order, a small e-connection step and a small parameter-shift update coincide: our shift operator is the e-covariant derivative of the Fisher metric written in e-affine coordinates. 

This is especially clean for the auxiliary score process. After Gaussianization and precision parameterization, that process is approximately a canonical exponential family, so the same calculation gives

$$ P_{\theta + d\theta \to \theta}^{(e)} \mathcal I_\delta(\theta + d\theta) = \mathcal I_\delta(\theta) + G_\delta(\theta, d\theta) + O(\|d\theta\|^2). $$

Returning from the Gaussian surrogate to the original score process only introduces the remainder already controlled above:

$$ P_{\theta + d\theta \to \theta}^{(e)} \mathcal I_\delta(\theta + d\theta) = \mathcal I_\delta(\theta) + G_\delta(\theta, d\theta) + B_\delta(\theta, d\theta) + O(\|d\theta\|^2), $$

$$ \| B_\delta(\theta, d\theta) \| \leq C (\sqrt{\pi_\delta} + \delta) \|d\theta\|. $$

So, the e-connection interpretation is exact for the Gaussianized auxiliary family and approximate for the underlying task model, with the approximation improving in the same regime that justifies our score-based sufficient statistic estimates. 

## Experimental considerations 

In contexts where $\theta_t$ moves incredibly slowly per many observations gathered, 
the math is clear: the Amari-Chentsov tensor will produce more-accurate FIM estimates. 
Of course, with even greater data volumes per $\theta_t$ step, 
we could just entirely re-estimate the FIM entirely and not need Amari-Chentsov updates at all. 
As the third term in the log likelihood's Taylor Series expansion, 
the Amari-Chentsov tensor is the expectation of triple outer-product of score vectors, 
a high-variance entity. 
Further, as a rank-3 tensor a naive representation will take $O(p^3)$ space per $p$ tunable model parameters, absolutely massive. 
So, there are statistical and numerical considerations the ultimately challenge the practical effectiveness of this technique. 
Hence, an applied experiment is begged. 

We will run an MNIST experiment with small models studying:
1. When do Amari-Chentsov updates add value beyond FIM moving average estimates? 
2. Which strategies are useful in controlling the high-variance of $\widehat{C : d\theta}$? 

We'll then run another MNIST experiment with 
(1) scalable low-rank approximations and 
(2) small batch sizes (as low as 1),
and attempt to reproduce results observed in our first round of MNIST experiments. 
If successful, this'll give us a foundation for applied success, because:
1. low-rank approximations of the FIM and Amari-Chentsov tensor keep space complexity manageably bounded for large models, and 
2. single observation batch updates drastically reduce VRAM consumed by activations while estimating the FIM and Amari-Chentsov tensor.   

We'll then conclude with a robotics demonstration illustrating how this technology can fine-tune a Visual Language Model (VLM) for specific application. 


## Mathematical Model

For every $\theta \in \Theta$ assume the existence of a pairing $d(\theta) \in \Theta$. 
Whenever a model is at $\theta$, it draws samples 
from $d(\theta)$. 
When we update our model with the new data to some $\widehat{d(\theta)}$, 
then we say the model is now at $\theta \gets \widehat{d(\theta)}$. 
Naturally, much of our work assumes $d\theta := d(\theta) - \theta$ is a small value. 

It is convenient to further assume the existence of a well-defined and finite gradient field $\mathbb E_\theta \nabla_\theta \ell$ 
and Hessians $\mathbb E_\theta \nabla_\theta^2 \ell$, Amari-Chentsov tensors $\mathbb E_\theta \nabla_\theta^3 \ell$, 
and fourth derivatives. 
These assumptions allow us to construct a coherent theory of Amari-Chentsov-updated EWC regularizers. For single-observation batches below, the corresponding first-order parameter-shift is instead carried by the auxiliary score process introduced above. 
If we further assume
(1) sufficiently many samples per $\theta_t$ to invoke CLT and thus Gaussian steps, and 
(2) choose $d(\theta)$ as a correct function of the gradient field $\mathbb E_\theta \nabla_\theta \ell$, 
then we enjoy the existence of a Stochastic Differential Equation (SDE) well-approximating our process over the statistical manifold. 
 
To derive the EWC regularizer under a frequentist paradigm, 
we introduce a mixture model according to _observed_ Bernoulli variable $M_t$, 
such that:
- $\mathbb P[ M_t = 1] = \pi = 1 - \mathbb P[M_t = 0]$, 
- $(X_t \, | \, M_t = 0) \sim f_X(x; \theta_t)$, and
- $(X_t \, | \, M_t = 1) \sim f_X(x; \theta_t + d\theta_t) $.

To reproduce the EWC regularizer, 
break our observation vector into two sub-vectors $X = (X^t, X^{t+1})$, where 
- $X_i^t = (X_i \, | \, M_i = 0)$, and
- $X_i^{t+1} = (X_i \, | \, M_i = 1)$. 

Now break the log likelihood into two parts.

$$ \log f_X(X;\theta + d\theta) = \log f_X(X^{t+1}; \theta + d\theta) + \log f_X(X^t ; \theta + d\theta) $$

$$ \approx \log f_X(X^{t+1}; \theta + d\theta) + \log f_X(X^t ; \theta) + d\theta^T \nabla_\theta \log f_X(X^t; \theta) + 2^{-1} d\theta \left(\nabla_\theta^2 \log f_X(X^t ; \theta)\right) d\theta $$

$$ \approx_{a.s.} \log f_X(X^{t+1}; \theta + d\theta) + 0 - 2^{-1} \left( \sum_{i=1}^n \mathbb I_{\{M_i = 0\}} \right) d\theta^T \mathcal I (\theta) d\theta $$

We've almost recovered EWC, 
but the regularizer constant takes an elegant $(1 - \pi)/2$ form under MLE optimization. 

$$ \hat \theta_{t+1} = \arg\max_{d\theta} n^{-1} \log f_X(X; \theta + d\theta) \approx n^{-1} \log f_X(X^{t+1}; \theta + d\theta) - \frac{1-\pi}{2} d\theta^T \mathcal I (\theta) d\theta $$

This has approximate asymptotic distribution $\hat \theta_{t+1} \sim \mathcal N \left( \pi d\theta_t + \theta_t, \; \pi \mathcal I^{-1}(\theta_t+d\theta_t) / n \right)$.


## SDE construction 

For each integer $k \geq 0$, there exists triangular array $\{ X_{i,n}^k \}_{i=1}^n$ of observations with known $M_{i,n}^k$ mixture variables. 
Assume the existence of a single gradient field $b(\theta)$ on $\Theta$, 
allowing us to coherently define stochastic integral paths over the statistical manifold. 
This gives us a clean asymptotic approximation:

$$ \theta_{k+1,n} \approx \theta_{k,n} + \pi b(\theta_{k,n})/n + \sqrt{\pi \mathcal I^{-1} (\theta_{k,n}) /n } \, \xi_k, \; \xi_k \sim_{iid} \mathcal N(0, \; I_p)$$

Scaling with $k = \lfloor nt \rfloor$ and $n \to \infty$ produces the SDE.

$$ d\Theta_t = \pi b(\Theta_t) dt + \sqrt{\pi \mathcal I^{-1} (\Theta_t)} dW_t $$


## Optimal $\pi$ as stochastic control 

A one-step MSE-optimal estimate of $\theta_{t+1}$ is obtained by choosing
$$ \pi_t^* = 1 - \mathrm{tr}\left[ \mathcal I^{-1}(\theta_t) n^{-1} \right] / \left( 2 \| d\theta_t \|^2 \right). $$
Indeed, writing $\theta_{t+1}^* := \theta_t + d\theta_t$ and using the Gaussian approximation above, the error enjoys approximate mean $(\pi - 1) d\theta_t$ and covariance $\pi \, \mathcal I^{-1}(\theta_t) / n$. Using $\mathbb E \|X\|^2 = \| \mathbb E X \|^2 + \mathrm{tr}(\mathrm{Cov}(X))$ then gives
$$ \mathbb E \| \hat \theta_{t+1} - \theta_{t+1}^* \|^2 \approx \| (\pi - 1) d\theta_t \|^2 + \pi \, \mathrm{tr}\left[ \mathcal I^{-1}(\theta_t) n^{-1} \right]. $$
Differentiating in $\pi$ and setting the derivative to zero gives the stated rule. 
This is naturally interpreted as a feedback rule for $\pi_t$, not for $\| d\theta_t \|$. 
Indeed, in application the true gap $d\theta_t$ belongs to the environment, while $\pi_t$ is the quantity we may choose. 

So, we instead choose a different limit and thus limiting process. 
A model at $\theta_{k,n}^*$ has true generating point $ \theta_{k,n}^* + d\theta_{k,n}^* = \theta_{k,n}^* + b(\theta_{k,n}^*) / \sqrt{n} $. 
Allowing $\pi_{k,n} \in [0,1]$ to vary adaptively gives the controlled update equation:

$$ \theta_{k+1,n}^* = \theta_{k,n}^* + \pi_{k,n} \frac{b(\theta_{k,n}^*)}{\sqrt n} + \sqrt{\pi_{k,n} \mathcal I^{-1}(\theta_{k,n}^*) / n} \, \xi_k $$

Taking $k = \lfloor \sqrt n t \rfloor$, and assuming $b$ and $\mathcal I^{-1}$ are Lipschitz continuous, 
we get a controlled discrete process with:
- $ \Delta_{k,n} := \theta_{k+1,n}^* - \theta_{k,n}^*$
- $ \Rightarrow \mathbb E [ \Delta_{k,n} \, | \, \mathcal F_k ] = \pi_{k,n} b(\theta_{k,n}^*) / \sqrt n $ and
- $ \mathrm{Cov} [ \Delta_{k,n} \, | \, \mathcal F_k ] = \pi_{k,n} \mathcal I^{-1}(\theta_{k,n}^*) / n $. 
- $ \sum_{j=1}^{k-1} \mathbb E [ \Delta_{j,n} \, | \, \mathcal F_j ] = \sum_{j=1}^{\lfloor \sqrt n t \rfloor-1} \pi_{j,n} b(\theta_{j,n}^*) / \sqrt n = O(\sqrt n / \sqrt n) $, a Riemann sum which converges by Lipschitz continuity.
- $ \sum_{j=1}^{k-1} \mathrm{Cov} [ \Delta_{j,n} \, | \, \mathcal F_j ] = \sum_{j=1}^{\lfloor \sqrt n t \rfloor-1} \pi_{j,n} \mathcal I^{-1}(\theta_{j,n}^*) / n = O( \sqrt n / n)$, so the stochastic term vanishes.

So, stochastic control still causes randomness to vanish with large sample sizes, 
leaving us with controlled path integral $\Theta_t^* = \Theta_0^* + \int_0^t \pi_s b(\Theta_s^*) ds$ 
or simply $\dot \Theta_t^* = \pi_t b( \Theta_t^*)$. 
Of course, no true sample size is ever infinite, 
so we may find it pragmatic to approximately model the discrete process with controlled small-noise SDE $\Theta_t^\varepsilon$:

$$ d\Theta_t^\varepsilon = \pi_t b(\Theta_t^\varepsilon) dt + \sqrt{\varepsilon \pi_t} \mathcal I^{-1/2}(\Theta_t^\varepsilon) dW_t, \; \varepsilon = n^{-1/2}.$$


## Why MNIST is mathematically comparable to Reinforcement Learning (RL) 

In RL, the agent at $\theta_t$ can be imagined to sample data from some unknown $\theta_{t+1}$.  is an abstract numerical experiment studying how tracking different sufficient statistics can most-optimally guide SDEs according to the theory
Upon a model update, the model _moves_ to $\theta_{t+1} \gets \hat \theta_{t+1}$. 
So, our model reduces RL to estimating a slowly moving target over a set of distributions. 
This, of course, invites distortions to our FIM estimate, so perhaps motivates Amari-Chentsov updates. 

Our MNIST experiment will initially fit a model to digits 0 through 8, inclusive. 
This initial model is fit on a large amount of data, 
so represents the EMC + LoRA context: fine tuning an initial large model. 
We will then start sampling 9s with some probability $p$, starting of course with $p = 0$. 
Then we will slowly increase $p \to 1/2$, until 9s compose about 50% of observed samples. 
Our primary metric: accuracy in classifying 9s. 

Both RL and our MNIST experiment, under our model, 
are estimating a slowly moving target $\theta_i$ from independent samples $X_i \sim f_X(x; \theta_i)$. 
In this sense, they are comparable. 


## Single Observation Batches 

Modern large deep nets with LoRA tend to have significantly larger activations than gradients in VRAM space requirements. 
By reformulating our framework to handle single observation updates to sufficient statistics, 
we drastically reduce VRAM requirements to a truly online scale: 
storing just the parameter, an observation's score, and the sufficient statistics. 

Rather than invoke CLT within a batch, we may Gaussianize over time with normalized exponential moving averages. Using the same scaling as above, let

$$ \theta_{t+1,\delta} = \theta_{t,\delta} + \pi b(\theta_{t,\delta}) / n_\delta + \sqrt{\pi / n_\delta} \, \mathcal I^{-1}(\theta_{t,\delta}) Y_{t,\delta}. $$

For scores $s_t := \nabla_\theta \ell(X_t; \theta_{t,\delta})$, define

$$ \bar s_t^{(\delta)} = (1 - \pi_\delta) \bar s_{t-1}^{(\delta)} + \pi_\delta s_t, \qquad Y_{t,\delta} := \sqrt{(2 - \pi_\delta)/\pi_\delta} \, \bar s_t^{(\delta)}. $$

Assume the score's third absolute moment is uniformly bounded, $b$ and $\mathcal I(\theta)$ are Lipschitz continuous, $n_\delta \to \infty$, $\pi_\delta \to 0$, and the parameter moves slowly enough over one EMA memory horizon that

$$ \Delta_{t,\delta} := \sum_{j \geq 0} \pi_\delta (1 - \pi_\delta)^j \| \theta_{t-j,\delta} - \theta_{t,\delta} \| = O(\delta). $$

For frozen $\theta$, the weighted Lindeberg-Feller theorem gives $Y_{t,\delta} \Rightarrow \mathcal N(0, \mathcal I(\theta))$ as $\pi_\delta \to 0$. Expanding around the local frozen parameter then yields an asymptotically centered Gaussian family with covariance $\mathcal I(\theta)$. Parameterized by precision, this Gaussian family is canonical exponential, so its first-order parameter-shift is score-only. Consequently $\mathcal I^{-1}(\theta) Y_{t,\delta}$ has approximate covariance $\mathcal I^{-1}(\theta)$, matching the large-batch scaling. Accordingly, with

$$ [B_\delta(\theta, u)]_{ij} := u^k \partial_k [\mathcal I_\delta(\theta)]_{ij} - [G_\delta(\theta, u)]_{ij}, $$

we get

$$ \| B_\delta(\theta, u) \| \leq C (\sqrt{\pi_\delta} + \delta) \| u \| \to 0, $$

and

$$ \mathcal I(\theta + u) = \mathcal I(\theta) + G_\delta(\theta, u) + B_\delta(\theta, u) + O(\|u\|^2). $$

This admits in-place sufficient statistic updates. Writing $u_t := \theta_{t+1,\delta} - \theta_{t,\delta}$, $Z_t := s_t s_t^T$, and $\Gamma_t := (u_t^T s_t) s_t s_t^T$, define

$$ \bar G_t = (1 - \pi_\delta) \bar G_{t-1} + \pi_\delta \Gamma_t, $$

$$ \bar{\mathcal I}_t = (1 - \pi_\delta) (\bar{\mathcal I}_{t-1} + \bar G_{t-1}) + \pi_\delta Z_t. $$

Then $\widehat{\mathcal I}^{\, \mathrm{shift}}_t := \bar{\mathcal I}_t + \bar G_t$ aims at $\mathcal I(\theta_{t,\delta} + u_t)$. Ignoring sampling noise, if $\bar{\mathcal I}_{t-1} = \mathcal I(\theta_{t-1,\delta})$ and $\bar G_{t-1} = G_\delta(\theta_{t-1,\delta}, u_{t-1})$, then $\bar{\mathcal I}_t = \mathcal I(\theta_{t,\delta}) + O(\| u_{t-1} \|^2)$, so the $G$-shift removes the usual first-order EMA lag. Finally, with $k = \lfloor n_\delta t \rfloor$ we recover the same limit form as above,

$$ d\Theta_t = \pi b(\Theta_t) dt + \sqrt{\pi} \, \mathcal I^{-1/2}(\Theta_t) dW_t. $$

## Optimal $\pi$ as stochastic control for single observation batches 

Here, $\pi$ is best interpreted as a control variable rather than a literal sample proportion. In application this is natural: as information is forgotten by our EWC-like machinery, the original pre-training sample size becomes progressively less relevant. In implementation one may also tie $\pi$ to the EMA gain $\pi_\delta$ above, though the two roles can be separated. As in the large-batch setting, we therefore offer two policies: a conservative user-chosen $\pi$, and a one-step MSE-optimal $\pi^*$ which is only locally optimal and may forget too aggressively for globally optimal control.

Let the true next generating point be

$$ \theta_{t+1,\delta}^* := \theta_{t,\delta} + d\theta_{t,\delta}. $$

Under the single-observation EMA construction above, the observed update is approximately Gaussian,

$$ \hat \theta_{t+1,\delta} - \theta_{t,\delta} \approx \mathcal N \left( \pi \, d\theta_{t,\delta}, \; \pi \, \mathcal I^{-1}(\theta_{t,\delta}) / n_\delta \right), $$

where the covariance approximation inherits the $O(\sqrt{\pi_\delta} + \delta)$ error discussed above. Repeating the large-batch calculation with $n$ replaced by $n_\delta$ gives

$$ \mathbb E \| \hat \theta_{t+1,\delta} - \theta_{t+1,\delta}^* \|^2 \approx \| (\pi - 1) d\theta_{t,\delta} \|^2 + \pi \, \mathrm{tr}\left[ \mathcal I^{-1}(\theta_{t,\delta}) / n_\delta \right]. $$

Differentiating in $\pi$ gives the regularized feedback rule

$$ \pi_{t,\delta}^* = 1 - \mathrm{tr}\left[ \mathcal I^{-1}(\theta_{t,\delta}) / n_\delta \right] / \left( 2 \| d\theta_{t,\delta} \|^2 + \varepsilon \right), \qquad \varepsilon > 0. $$

In practice one may clip this to $[0,1]$, or blend the two policies with $\lambda \pi_{t,\delta}^* + (1-\lambda) \pi$ for some $\lambda \in [0,1]$. The former is locally MSE-optimal, while the latter may better serve globally optimal control. 

As above, this suggests a controlled discrete process. A model at $\theta_{k,n_\delta}^{*,\delta}$ has true generating point $\theta_{k,n_\delta}^{*,\delta} + d\theta_{k,n_\delta}^{*,\delta} = \theta_{k,n_\delta}^{*,\delta} + b(\theta_{k,n_\delta}^{*,\delta}) / \sqrt{n_\delta}$, and the observed update obeys the approximate recursion

$$ \theta_{k+1,n_\delta}^{*,\delta} = \theta_{k,n_\delta}^{*,\delta} + \pi_{k,\delta} \frac{b(\theta_{k,n_\delta}^{*,\delta})}{\sqrt{n_\delta}} + \sqrt{\pi_{k,\delta} \, \mathcal I^{-1}(\theta_{k,n_\delta}^{*,\delta}) / n_\delta} \, \xi_{k,\delta}, \qquad \xi_{k,\delta} \approx \mathcal N(0, I_p), $$

where the Gaussian approximation again inherits $O(\sqrt{\pi_\delta} + \delta)$ error. Taking $k = \lfloor \sqrt{n_\delta} t \rfloor$ still causes the stochastic term to vanish asymptotically, so for finite data it is pragmatic to model the process by the small-noise SDE

$$ d\Theta_t^{\varepsilon,\delta} = \pi_t b(\Theta_t^{\varepsilon,\delta}) dt + \sqrt{\varepsilon \pi_t} \, \mathcal I^{-1/2}(\Theta_t^{\varepsilon,\delta}) dW_t, \qquad \varepsilon = n_\delta^{-1/2}. $$

## When local optimality demands forgetting

The one-step optimal rule $\pi^*$ should not necessarily be interpreted as a globally desirable learning policy. A large value of $\hat \pi^*$ indicates that, under the local MSE approximation, the current system change $\|d\hat\theta_{t,\delta}\|$ is large relative to the local statistical uncertainty

$$ \mathrm{tr}\left[\widehat{\mathcal I}^{-1}(\theta_{t,\delta}) / n_\delta\right]. $$

Locally, this makes heavy weighting of the present appear optimal. However, in the EMA update for the Fisher estimate,

$$ \bar{\mathcal I}_t = (1 - \pi_\delta) (\bar{\mathcal I}_{t-1} + \bar G_{t-1}) + \pi_\delta Z_t, $$

larger $\pi_\delta$ values rapidly discount old information. After $m$ steps, information carried by a previous estimate is weighted by approximately $(1 - \pi_\delta)^m$. Thus, if the EMA gain is tied too closely to $\hat\pi^*$, then large apparent changes in the world can cause the agent to quickly discard previously accumulated curvature information. This is a form of catastrophic forgetting: the locally optimal response to significant change may be globally destructive.

For this reason, $\hat\pi^*$ may be more useful as a diagnostic than as a direct control rule. A large $\hat\pi^*$ can be interpreted as evidence that the current online learning regime is being asked to absorb too much change too quickly. In this case, the correct response may not be to increase plasticity, but to protect memory and reduce the effective rate of adaptation.

A practical policy is to choose a maximum allowed forgetting rate $\pi_{\max} \in (0,1)$ and enforce

$$ \pi_{\mathrm{used}} = \min(\hat\pi^*, \pi_{\max}). $$

When $\hat\pi^*$ exceeds this cap, the experiment should be treated as outside the ordinary online adaptation regime. Possible interventions include reducing the learning rate, increasing the EMA memory horizon, increasing batch size, replaying older data, strengthening the EWC penalty, freezing part of the model, or slowing the environmental change itself.

Under this interpretation, $\hat\pi^*$ acts as an overwhelm diagnostic. High values indicate that the locally optimal estimator would need to forget too aggressively in order to track the present change. Rather than treating this as permission to overwrite memory, the system should treat it as a warning that the agent is at risk of over-weighting a transient shock and losing useful long-run structure.


## Citations 

[1] Y. Zheng, Y. Zhang, J. van de Weijer, G. M. van de Ven, S. Du, X. Zhang, and Z. Tian, [*Revisiting Weight Regularization for Low-Rank Continual Learning*](https://arxiv.org/abs/2602.17559), arXiv:2602.17559, 2026.

[2] S. Amari and H. Nagaoka, *Methods of Information Geometry*, American Mathematical Society, 2000. 